# 4x4 Fermi-Hubbard Starter: Direct Fermionic Pepsy/Symmray

This notebook is a small starting point for direct fermionic Fermi-Hubbard simulations in Pepsy/Symmray. The main methods reference is Gao et al., *Fermionic tensor network contraction for arbitrary geometries*, Phys. Rev. Research 7, 023193 (2025), https://doi.org/10.1103/PhysRevResearch.7.023193. The square-lattice physics settings are taken from arXiv:2511.02125, whose hardware workflow uses a fermion-to-qubit encoding; this notebook does **not** use that mapping. It works directly with fermionic Symmray tensor-network objects through Pepsy.

The first target is intentionally modest: build a 4x4 spinful fermionic PEPS at half filling, construct the Hubbard Hamiltonian with `t = 1` and `U/t = 8`, inspect the block-sparse structure, and optionally run one tiny imaginary-time step.

In [ ]:
import math
import os

os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache_pepsy")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig_pepsy")
os.makedirs(os.environ["NUMBA_CACHE_DIR"], exist_ok=True)
os.makedirs(os.environ["MPLCONFIGDIR"], exist_ok=True)

import numpy as np

import pepsy as py

try:
    import symmray as sr
except ImportError as exc:
    raise ImportError(
        "This example needs the optional dependency `symmray`. "
        "Install it in the Pepsy environment before running the notebook."
    ) from exc

print("pepsy", py.__version__)
print("symmray", getattr(sr, "__version__", "installed"))

## Model Settings

The paper's half-filled Hubbard section uses a periodic 6x6 square lattice and mixed periodic boundary-condition averaging. For a first direct-fermion tensor-network smoke test, this notebook starts with a 4x4 lattice. `CYCLIC = False` keeps the first contraction cheaper; set it to `True` once the small open-boundary workflow runs cleanly.

In [ ]:
Lx = 4
Ly = 4
t = 1.0
U = 8.0
D = 2
CYCLIC = False
SEED = 251102125

num_sites = Lx * Ly
num_spin_orbitals = 2 * num_sites
target_up = num_sites // 2
target_down = num_sites // 2
target_particles = target_up + target_down
target_charge = (target_up, target_down)

phys_sectors = py.default_physical_sectors(model="fermi_hubbard_u1u1")
half_filled_charges = {
    (x, y): (1, 0) if (x + y) % 2 == 0 else (0, 1)
    for x in range(Lx)
    for y in range(Ly)
}
site_charge = py.site_charge_from_occupations(half_filled_charges)

print("local sectors:", phys_sectors)
print("sites:", num_sites)
print("spin orbitals:", num_spin_orbitals)
print("target total particle number:", target_particles)
print("target (N_up, N_down):", target_charge)
print("filling N / (2 sites):", target_particles / num_spin_orbitals)
print("boundary:", "periodic" if CYCLIC else "open smoke test")

## Half Filling And Symmetry Sector

For the spinful Hubbard model, each site has two spin orbitals. Half filling means `N_particles = N_sites`, not `2 * N_sites`. Here that is 16 particles on 16 lattice sites, or 16 particles in 32 spin orbitals.

The symmetry used below is spin-resolved `U1U1`, with local charges `(n_up, n_down)`. The notebook fixes `N_up = 8` and `N_down = 8`, so the total particle number is `16` on `16` lattice sites. The simple alternating charge pattern is only a sector choice for initialization; it is not a fermion-to-qubit mapping.

## Build A Direct Fermionic PEPS

Each lattice site has the spinful Hubbard local basis with charge sectors `(0, 0)`, `(0, 1)`, `(1, 0)`, and `(1, 1)`. The `site_charge` map fixes the global `U1U1` sector to `(8, 8)`, i.e. 16 fermions on this 4x4 lattice.

In [ ]:
psi = py.SymPEPS.random(
    Lx,
    Ly,
    symmetry="U1U1",
    fermionic=True,
    phys_dim=phys_sectors,
    site_charge=site_charge,
    bond_dim=D,
    cyclic=CYCLIC,
    seed=SEED,
    dtype="complex128",
    contraction_opt="auto-hq",
)

ham = py.SymHamiltonian.from_edges(
    "fermi_hubbard_u1u1",
    "U1U1",
    psi.edges,
    t=t,
    U=U,
    mu=0.0,
)

summary = py.symmray_peps_summary(psi)

print("sites:", psi.num_sites)
print("edges / local Hamiltonian terms:", len(ham.terms))
print("fermionic tensors:", psi.fermionic)
print("total U1U1 charge:", psi.overall_charge())
print("max PEPS bond dim:", summary["max_bond_dim"])
print("stored / dense entries:", summary["total_stored_size"], "/", summary["total_dense_size"])
print("block storage density:", f"{summary['density']:.3f}")

## Same Half-Filled Sector As An MPS

This builds the same direct-fermion half-filled 4x4 problem as an MPS. The MPS uses a snake ordering for the sites; the Hamiltonian still contains the 2D square-lattice edges, so some gates are nonlocal in the 1D order.

In [ ]:
snake_sites = []
for x in range(Lx):
    ys = range(Ly) if x % 2 == 0 else range(Ly - 1, -1, -1)
    for y in ys:
        snake_sites.append((x, y))

coord_to_mps = {site: i for i, site in enumerate(snake_sites)}
mps_edges = tuple(
    (coord_to_mps[a], coord_to_mps[b])
    for a, b in psi.edges
)

psi_mps = py.SymMPS.random(
    num_sites,
    symmetry="U1U1",
    fermionic=True,
    phys_dim=phys_sectors,
    site_charge=py.site_charge_from_occupations([
        half_filled_charges[site]
        for site in snake_sites
    ]),
    bond_dim=D,
    seed=SEED,
    dtype="complex128",
)
ham_mps = py.SymHamiltonian.from_edges(
    "fermi_hubbard_u1u1",
    "U1U1",
    mps_edges,
    t=t,
    U=U,
    mu=0.0,
)
mps_summary = py.symmray_mps_summary(psi_mps)

print("MPS sites:", psi_mps.num_sites)
print("MPS total U1U1 charge:", psi_mps.overall_charge())
print("MPS Hamiltonian terms:", len(ham_mps.terms))
print("MPS max bond dim:", mps_summary["max_bond_dim"])
print("first snake sites:", snake_sites[:6])

## Inspect One Fermionic Hubbard Term

This checks that the Hamiltonian terms are Symmray block-sparse arrays, not qubit Pauli operators.

In [ ]:
edge, term = next(iter(ham.terms.items()))
term_summary = py.symmray_block_summary(term)

print("example edge:", edge)
print("term type:", type(term))
print("term shape:", term_summary["shape"])
print("number of stored blocks:", term_summary["num_blocks"])
print("stored / dense entries:", term_summary["stored_size"], "/", term_summary["dense_size"])
term_summary["blocks"][:8]

## Optional Energy Smoke Test

For a 4x4 PEPS this exact local-energy contraction can still be nontrivial. Keep `COMPUTE_ENERGY = False` while editing quickly, then turn it on for the first numerical check.

In [ ]:
COMPUTE_ENERGY = False

if COMPUTE_ENERGY:
    e0 = psi.energy_density(ham)
    print("initial <H>/N:", np.real_if_close(e0))
    print("initial <H>/N - U/4:", np.real_if_close(e0 - U / 4))
else:
    print("Set COMPUTE_ENERGY = True to contract the initial energy density.")

## Optional Tiny Imaginary-Time Step

The paper uses optimized preparation circuits before probing eta pairing. As a direct-fermion TN starting point, a small imaginary-time step is a simpler first sanity check. It is not meant to reproduce the paper's optimized state yet.

In [ ]:
RUN_ONE_IMAGINARY_TIME_STEP = False

if RUN_ONE_IMAGINARY_TIME_STEP:
    psi_it = psi.ground_state(
        dt=0.01,
        steps=1,
        hamiltonian=ham,
        order=1,
        max_bond=4,
        cutoff=1.0e-10,
        method="gate",
        inplace=False,
    )
    print("after one step max bond:", psi_it.tn.max_bond())
    print("after one step norm:", np.real_if_close(psi_it.norm()))
else:
    psi_it = None
    print("Set RUN_ONE_IMAGINARY_TIME_STEP = True to run one tiny projection step.")

## Paper Light-Pulse Constants

These constants are included here so the next notebook iteration can add Peierls phases directly to fermionic hopping terms, still without any fermion-to-qubit mapping.

In [ ]:
omega = 4 * math.pi / 3
tau = math.pi / (2 * omega)

def vector_potential(s):
    return math.pi * (1 - math.cos(omega * s)) / 2

print("omega:", omega)
print("tau:", tau)
print("pulse end time pi/omega:", math.pi / omega)
print("A(0):", vector_potential(0.0))
print("A(pi/omega):", vector_potential(math.pi / omega))

## Doped Checkerboard Targets From The Paper

For the next direct-fermion notebook, the doped checkerboard case should use a 6x6 lattice with spin-resolved charge `(15, 15)`. That is 30 fermions on 36 sites, i.e. six holes relative to half filling. The paper gives two useful comparison numbers at the weak-coupling checkerboard point `t' = 0`, `U/t = 2`: exact `<H>/N - U/4 = -1.27367`, and theoretical d-wave average `0.108`. The experimental d-wave average is `0.079 +/- 0.005`.

In [ ]:
doped_Lx = 6
doped_Ly = 6
doped_sites = doped_Lx * doped_Ly
doped_target_charge = (15, 15)
doped_holes = doped_sites - sum(doped_target_charge)
checkerboard_reference_energy_shifted = -1.27367
checkerboard_theory_dwave_average = 0.108
checkerboard_experiment_dwave_average = (0.079, 0.005)

print("doped checkerboard target charge:", doped_target_charge)
print("holes relative to half filling:", doped_holes)
print("reference <H>/N - U/4:", checkerboard_reference_energy_shifted)
print("theory d-wave average:", checkerboard_theory_dwave_average)

## Next Direct-Fermion Steps

- Add weighted or phased Fermi-Hubbard terms so the Peierls light pulse acts directly on hopping edges.
- Add onsite eta-pair correlators `Delta_i^dag Delta_j + h.c.` as fermionic Symmray observables.
- Repeat the 4x4 run with `CYCLIC = True`, then compare against a small exact reference.
- Only after the 4x4 checks are stable, move the same workflow toward the paper's 6x6 half-filled setting.